# 05 · Explainable AI (scaffold)
SHAP attributions for fitted models. **SHAP explains the model, not the ocean** — do not interpret attributions causally; correlated lags share credit arbitrarily.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline
from src import modelling, explainability

In [ ]:
RUN_TRAINING = True

In [ ]:
import pandas as pd
s = pd.read_csv('../data/processed/thetao_study_area_monthly.csv', index_col=0, parse_dates=True).iloc[:, 0]
m = settings['modelling']
split = modelling.chronological_split(modelling.make_features(s, m['lags']),
                                      m['train_end'], m['validation_end'], m['test_end'])
X = modelling.feature_cols(split.train)

In [ ]:
if RUN_TRAINING:
    imp = {}
    for name in ('xgboost', 'random_forest'):
        model = modelling.build_models(settings['project']['random_seed'])[name]
        model.fit(split.train[X], split.train['y'])
        exp = explainability.shap_values(model, split.validation[X])
        imp[name] = explainability.mean_abs_importance(exp, X)
    imp = pd.DataFrame(imp).round(3)
    imp.to_csv('../outputs/tables/05_shap_importance.csv')
    display(imp)

In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt
    fig = explainability.summary_plot(exp, split.validation[X])
    visualisation.save(plt.gcf(), '05_shap_summary_random_forest')